# Lab 4: Deploy Medical AI Agent

AgentCore Runtime에 Medical AI Agent를 배포하고 Streamlit UI로 연결합니다.

## Architecture
```
User (Browser) → Streamlit (EC2:8501)
  → boto3 InvokeAgentRuntime
    → AgentCore Runtime (medical_agent.py + Strands + Bedrock Claude Sonnet 4.6)
      → AgentCore Gateway (OAuth) → MCP Server (Lambda) → EMR Serverless → S3 Tables
      → AgentCore Memory (STM + LTM)
      → AgentCore Observability (CloudWatch + X-Ray)
```

## Step 1: Configuration

In [ ]:
import boto3
import json
import time
import os
import zipfile
import io

REGION = (
    boto3.session.Session().region_name
    or os.environ.get("AWS_DEFAULT_REGION")
    or os.environ.get("AWS_REGION")
)
sts = boto3.client("sts")
ACCOUNT_ID = sts.get_caller_identity()["Account"]
AGENT_NAME = "fhir_medical_agent"
S3_BUCKET = f"fhir-data-{ACCOUNT_ID}-{REGION}"

# Load Gateway connection info from Lab 2
with open("connection_info.json") as f:
    conn = json.load(f)

GATEWAY_MCP_URL = conn["mcp_server_url"]
GATEWAY_CLIENT_ID = conn["client_id"]
GATEWAY_CLIENT_SECRET = conn["client_secret"]
GATEWAY_TOKEN_URL = conn["token_url"]

iam_client = boto3.client("iam")
s3_client = boto3.client("s3", region_name=REGION)
agentcore = boto3.client("bedrock-agentcore-control", region_name=REGION)

print(f"Account: {ACCOUNT_ID}")
print(f"Gateway: {GATEWAY_MCP_URL}")

Account: 997924005000
Gateway: https://fhir-medical-gateway-zrar3kofie.gateway.bedrock-agentcore.us-west-2.amazonaws.com/mcp


/home/participant/.local/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


## Step 2: Enable CloudWatch Transaction Search (Observability)

AgentCore Observability를 위해 CloudWatch Transaction Search를 활성화합니다. (계정당 1회)

In [2]:
logs_client = boto3.client("logs", region_name=REGION)
xray_client = boto3.client("xray", region_name=REGION)

try:
    policy_doc = json.dumps({
        "Version": "2012-10-17",
        "Statement": [{
            "Sid": "TransactionSearchXRayAccess",
            "Effect": "Allow",
            "Principal": {"Service": "xray.amazonaws.com"},
            "Action": "logs:PutLogEvents",
            "Resource": [
                f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:aws/spans:*",
                f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:/aws/application-signals/data:*"
            ],
            "Condition": {
                "ArnLike": {"aws:SourceArn": f"arn:aws:xray:{REGION}:{ACCOUNT_ID}:*"},
                "StringEquals": {"aws:SourceAccount": ACCOUNT_ID}
            }
        }]
    })
    logs_client.put_resource_policy(policyName="AgentCoreTransactionSearch", policyDocument=policy_doc)
    print("✅ CloudWatch resource policy created")
except Exception as e:
    print(f"Resource policy: {e}")

try:
    xray_client.update_trace_segment_destination(Destination="CloudWatchLogs")
    print("✅ Transaction Search enabled")
except Exception as e:
    print(f"Transaction Search: {e}")

✅ CloudWatch resource policy created
Transaction Search: An error occurred (InvalidRequestException) when calling the UpdateTraceSegmentDestination operation: The destination is already set to CloudWatchLogs


## Step 3: Create AgentCore Memory (STM + LTM)

에이전트의 대화 기억을 위한 Memory 리소스를 생성합니다.

In [3]:
import uuid

memory_response = agentcore.create_memory(
    name=f"fhir_agent_mem_{uuid.uuid4().hex[:8]}",
    eventExpiryDuration=30,
    memoryStrategies=[
        {"userPreferenceMemoryStrategy": {
            "name": "user_prefs",
            "namespaces": ["/users/preferences/"]
        }},
        {"semanticMemoryStrategy": {
            "name": "medical_facts",
            "namespaces": ["/users/facts/"]
        }}
    ]
)

MEMORY_ID = memory_response["memory"]["id"]
print(f"Memory ID: {MEMORY_ID}")
print(f"Status: {memory_response['memory']['status']}")

# Wait for ACTIVE
print("Waiting for memory to become ACTIVE...")
for _ in range(60):
    mem_status = agentcore.get_memory(memoryId=MEMORY_ID)["memory"]["status"]
    if mem_status == "ACTIVE":
        print(f"✅ Memory is ACTIVE: {MEMORY_ID}")
        break
    time.sleep(5)
else:
    print(f"⚠️ Memory status: {mem_status}")

Memory ID: fhir_agent_mem_b0dc923e-HZw5kFBSwy
Status: CREATING
Waiting for memory to become ACTIVE...
✅ Memory is ACTIVE: fhir_agent_mem_b0dc923e-HZw5kFBSwy


## Step 4: Create IAM Role for AgentCore Runtime

Runtime이 Bedrock 모델 호출, Memory 접근, Gateway 호출 등에 사용할 IAM Role을 생성합니다.

In [4]:
RUNTIME_ROLE_NAME = "fhir-agentcore-runtime-role"

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
        "Action": "sts:AssumeRole"
    }]
}

# Create role
try:
    role = iam_client.create_role(
        RoleName=RUNTIME_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="AgentCore Runtime role for FHIR Medical Agent"
    )
    print(f"Created role: {RUNTIME_ROLE_NAME}")
except iam_client.exceptions.EntityAlreadyExistsException:
    print(f"Role exists: {RUNTIME_ROLE_NAME}")

RUNTIME_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{RUNTIME_ROLE_NAME}"

# Attach permissions
runtime_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "BedrockModelAccess",
            "Effect": "Allow",
            "Action": ["bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream"],
            "Resource": "*"
        },
        {
            "Sid": "AgentCoreMemory",
            "Effect": "Allow",
            "Action": ["bedrock-agentcore:*"],
            "Resource": "*"
        },
        {
            "Sid": "S3CodeAccess",
            "Effect": "Allow",
            "Action": ["s3:GetObject", "s3:GetBucketLocation"],
            "Resource": [
                f"arn:aws:s3:::{S3_BUCKET}",
                f"arn:aws:s3:::{S3_BUCKET}/*"
            ]
        },
        {
            "Sid": "CloudWatchLogs",
            "Effect": "Allow",
            "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents"],
            "Resource": "*"
        },
        {
            "Sid": "XRayTracing",
            "Effect": "Allow",
            "Action": ["xray:PutTraceSegments", "xray:PutTelemetryRecords"],
            "Resource": "*"
        },
        {
            "Sid": "ECRAccess",
            "Effect": "Allow",
            "Action": ["ecr:GetAuthorizationToken", "ecr:BatchGetImage", "ecr:GetDownloadUrlForLayer", "ecr:BatchCheckLayerAvailability"],
            "Resource": "*"
        }
    ]
}

iam_client.put_role_policy(
    RoleName=RUNTIME_ROLE_NAME,
    PolicyName="AgentCoreRuntimePolicy",
    PolicyDocument=json.dumps(runtime_policy)
)
print(f"✅ Role ARN: {RUNTIME_ROLE_ARN}")
time.sleep(10)  # Wait for IAM propagation

Role exists: fhir-agentcore-runtime-role
✅ Role ARN: arn:aws:iam::997924005000:role/fhir-agentcore-runtime-role


## Step 5: Build & Push Container Image

CodeBuild를 사용하여 에이전트 Docker 이미지를 빌드하고 ECR에 푸시합니다.

In [5]:
# --- Step 5: Build container with CodeBuild and push to ECR ---
import zipfile, io

ECR_REPO_NAME = "fhir-medical-agent"
ecr_client = boto3.client("ecr", region_name=REGION)
codebuild_client = boto3.client("codebuild", region_name=REGION)

# 1. Create ECR repo
try:
    ecr_client.create_repository(repositoryName=ECR_REPO_NAME)
    print(f"✅ ECR repo created: {ECR_REPO_NAME}")
except ecr_client.exceptions.RepositoryAlreadyExistsException:
    print(f"ECR repo exists: {ECR_REPO_NAME}")

ECR_URI = f"{ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/{ECR_REPO_NAME}:latest"
print(f"ECR URI: {ECR_URI}")

# 2. Upload agent source to S3 for CodeBuild
agent_dir = "../agent"
zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in ["medical_agent.py", "system_prompt.md", "requirements.txt", "Dockerfile"]:
        filepath = os.path.join(agent_dir, fname)
        if os.path.exists(filepath):
            zf.write(filepath, fname)
            print(f"  Added: {fname}")
    # Add buildspec.yml inline
    buildspec = """version: 0.2
phases:
  pre_build:
    commands:
      - aws ecr get-login-password --region $AWS_DEFAULT_REGION | docker login --username AWS --password-stdin $ECR_REGISTRY
  build:
    commands:
      - docker build -t $ECR_REGISTRY/$ECR_REPO:latest .
  post_build:
    commands:
      - docker push $ECR_REGISTRY/$ECR_REPO:latest
"""
    zf.writestr("buildspec.yml", buildspec)

zip_buffer.seek(0)
s3_key = "agent-code/agent-source.zip"
s3_client.put_object(Bucket=S3_BUCKET, Key=s3_key, Body=zip_buffer.getvalue())
print(f"\n✅ Source uploaded to s3://{S3_BUCKET}/{s3_key}")

# 3. Create CodeBuild service role
CB_ROLE_NAME = "fhir-agent-codebuild-role"
cb_trust = {
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Principal": {"Service": "codebuild.amazonaws.com"}, "Action": "sts:AssumeRole"}]
}
try:
    iam_client.create_role(RoleName=CB_ROLE_NAME, AssumeRolePolicyDocument=json.dumps(cb_trust))
except iam_client.exceptions.EntityAlreadyExistsException:
    pass

iam_client.put_role_policy(RoleName=CB_ROLE_NAME, PolicyName="CodeBuildPolicy", PolicyDocument=json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {"Effect": "Allow", "Action": ["ecr:*"], "Resource": "*"},
        {"Effect": "Allow", "Action": ["ecr:GetAuthorizationToken"], "Resource": "*"},
        {"Effect": "Allow", "Action": ["s3:GetObject", "s3:GetBucketLocation"], "Resource": [f"arn:aws:s3:::{S3_BUCKET}", f"arn:aws:s3:::{S3_BUCKET}/*"]},
        {"Effect": "Allow", "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents"], "Resource": "*"}
    ]
}))
CB_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{CB_ROLE_NAME}"
time.sleep(10)

# 4. Create/update CodeBuild project
CB_PROJECT = "fhir-medical-agent-build"
ECR_REGISTRY = f"{ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com"
project_config = dict(
    name=CB_PROJECT,
    source={"type": "S3", "location": f"{S3_BUCKET}/{s3_key}"},
    artifacts={"type": "NO_ARTIFACTS"},
    environment={
        "type": "ARM_CONTAINER",
        "image": "aws/codebuild/amazonlinux2-aarch64-standard:3.0",
        "computeType": "BUILD_GENERAL1_SMALL",
        "privilegedMode": True,
        "environmentVariables": [
            {"name": "ECR_REGISTRY", "value": ECR_REGISTRY},
            {"name": "ECR_REPO", "value": ECR_REPO_NAME},
        ]
    },
    serviceRole=CB_ROLE_ARN,
)
try:
    codebuild_client.create_project(**project_config)
    print(f"✅ CodeBuild project created: {CB_PROJECT}")
except codebuild_client.exceptions.ResourceAlreadyExistsException:
    codebuild_client.update_project(**project_config)
    print(f"CodeBuild project updated: {CB_PROJECT}")

# 5. Start build
build = codebuild_client.start_build(projectName=CB_PROJECT)
build_id = build["build"]["id"]
print(f"\n🔨 Build started: {build_id}")
print("Waiting for build to complete (~2-3 min)...")

for i in range(60):
    b = codebuild_client.batch_get_builds(ids=[build_id])["builds"][0]
    bs = b["buildStatus"]
    if bs == "SUCCEEDED":
        print(f"\n✅ Build SUCCEEDED! Image: {ECR_URI}")
        break
    elif bs in ("FAILED", "FAULT", "STOPPED"):
        print(f"\n❌ Build {bs}")
        phases = b.get("phases", [])
        for p in phases:
            if p.get("phaseStatus") == "FAILED":
                print(f"  Failed phase: {p['phaseType']} - {p.get('contexts', [{}])[0].get('message', '')}")
        break
    print(f"  {bs} ({i*10}s)", end="\r")
    time.sleep(10)

ECR repo exists: fhir-medical-agent
ECR URI: 997924005000.dkr.ecr.us-west-2.amazonaws.com/fhir-medical-agent:latest
  Added: medical_agent.py
  Added: system_prompt.md
  Added: requirements.txt
  Added: Dockerfile

✅ Source uploaded to s3://fhir-data-997924005000-us-west-2/agent-code/agent-source.zip
CodeBuild project updated: fhir-medical-agent-build

🔨 Build started: fhir-medical-agent-build:68dfe082-6ea5-4a67-82db-aa44fd5529b1
Waiting for build to complete (~2-3 min)...
  IN_PROGRESS (80s)
✅ Build SUCCEEDED! Image: 997924005000.dkr.ecr.us-west-2.amazonaws.com/fhir-medical-agent:latest


## Step 6: Create AgentCore Runtime

에이전트를 AgentCore Runtime에 배포합니다.

In [ ]:
# Environment variables for the agent
env_vars = {
    "AWS_REGION": REGION,
    "BEDROCK_AGENTCORE_MEMORY_ID": MEMORY_ID,
    "MODEL_ID": "us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    "GATEWAY_MCP_URL": GATEWAY_MCP_URL,
    "GATEWAY_CLIENT_ID": GATEWAY_CLIENT_ID,
    "GATEWAY_CLIENT_SECRET": GATEWAY_CLIENT_SECRET,
    "GATEWAY_TOKEN_URL": GATEWAY_TOKEN_URL,
    "GATEWAY_SCOPE": "fhir-mcp/tools",
}

try:
    runtime = agentcore.create_agent_runtime(
        agentRuntimeName=AGENT_NAME,
        agentRuntimeArtifact={
            "containerConfiguration": {
                "containerUri": ECR_URI
            }
        },
        roleArn=RUNTIME_ROLE_ARN,
        networkConfiguration={"networkMode": "PUBLIC"},
        environmentVariables=env_vars,
        description="FHIR Medical AI Agent with MCP Gateway integration",
    )

    AGENT_RUNTIME_ARN = runtime["agentRuntimeArn"]
    AGENT_RUNTIME_ID = runtime["agentRuntimeId"]
    print(f"✅ Runtime created!")
    print(f"   ARN: {AGENT_RUNTIME_ARN}")
    print(f"   ID: {AGENT_RUNTIME_ID}")
    print(f"   Status: {runtime['status']}")

except agentcore.exceptions.ConflictException:
    # Already exists — update with new code
    runtimes = agentcore.list_agent_runtimes()
    for r in runtimes.get("agentRuntimeSummaries", []):
        if r["agentRuntimeName"] == AGENT_NAME:
            AGENT_RUNTIME_ARN = r["agentRuntimeArn"]
            AGENT_RUNTIME_ID = r["agentRuntimeId"]
            break
    print(f"Runtime exists, updating: {AGENT_RUNTIME_ID}")
    runtime = agentcore.update_agent_runtime(
        agentRuntimeId=AGENT_RUNTIME_ID,
        agentRuntimeArtifact={
            "containerConfiguration": {
                "containerUri": ECR_URI
            }
        },
        roleArn=RUNTIME_ROLE_ARN,
        networkConfiguration={"networkMode": "PUBLIC"},
        environmentVariables=env_vars,
    )
    print(f"✅ Runtime updated! Status: {runtime.get('status')}")

✅ Runtime created!
   ARN: arn:aws:bedrock-agentcore:us-west-2:997924005000:runtime/fhir_medical_agent-8G4zZ2B28V
   ID: fhir_medical_agent-8G4zZ2B28V
   Status: CREATING


## Step 7: Wait for Runtime to be READY

In [7]:
print("Waiting for runtime to become READY...")
for i in range(60):
    status = agentcore.get_agent_runtime(agentRuntimeId=AGENT_RUNTIME_ID)
    current = status["status"]
    if current == "READY":
        print(f"\n✅ Runtime is READY! (took ~{i*10}s)")
        break
    elif "FAILED" in current:
        print(f"\n❌ Runtime FAILED: {current}")
        print(json.dumps(status, indent=2, default=str))
        break
    print(f"  Status: {current} ({i*10}s elapsed)")
    time.sleep(10)
else:
    print(f"⚠️ Timeout. Current status: {current}")

Waiting for runtime to become READY...

✅ Runtime is READY! (took ~0s)


## Step 8: Create Runtime Endpoint

In [8]:
try:
    endpoint = agentcore.create_agent_runtime_endpoint(
        agentRuntimeId=AGENT_RUNTIME_ID,
        name="DEFAULT",
        agentRuntimeVersion=str(status.get("agentRuntimeVersion", "1")),
    )
    print(f"✅ Endpoint created")
except agentcore.exceptions.ConflictException:
    print("Endpoint already exists, skipping creation")

# Wait for endpoint
print("\nWaiting for endpoint to become READY...")
for i in range(60):
    ep = agentcore.get_agent_runtime_endpoint(
        agentRuntimeId=AGENT_RUNTIME_ID,
        endpointName="DEFAULT"
    )
    ep_status = ep["status"]
    if ep_status == "READY":
        print(f"\n✅ Endpoint READY! (took ~{i*10}s)")
        break
    elif "FAILED" in ep_status:
        print(f"\n❌ Endpoint FAILED: {ep_status}")
        break
    print(f"  Status: {ep_status} ({i*10}s elapsed)")
    time.sleep(10)

Endpoint already exists, skipping creation

Waiting for endpoint to become READY...

✅ Endpoint READY! (took ~0s)


## Step 9: Test Agent — 기본 질문

In [52]:
runtime_client = boto3.client("bedrock-agentcore", region_name=REGION)

def invoke_agent(prompt, session_id=None):
    import uuid
    if not session_id:
        session_id = str(uuid.uuid4()) + "-" + "a" * 10
    
    payload = json.dumps({"prompt": prompt}).encode("utf-8")
    
    response = runtime_client.invoke_agent_runtime(
        agentRuntimeArn=AGENT_RUNTIME_ARN,
        payload=payload,
        runtimeSessionId=session_id,
        contentType="application/json",
        accept="application/json",
    )
    
    raw = response["response"].read()
    content_type = response.get("contentType", "")
    
    if "application/json" in content_type:
        body = json.loads(raw)
        return body.get("response", str(body))
    return raw.decode("utf-8")

print("Testing: 테이블 목록 조회")
print("=" * 60)
result = invoke_agent("데이터 레이크에 어떤 테이블들이 있는지 알려줘")
print(result)

Testing: 테이블 목록 조회
데이터 레이크에는 총 **24개 테이블**이 있으며, 5개 도메인으로 분류됩니다.

---

## 📋 데이터 레이크 테이블 목록

### 🏥 Administrative (행정/관리) — 5개
| 테이블명 | FHIR 리소스 | 설명 | 컬럼 수 |
|---|---|---|---|
| `patient` | Patient | 환자 인구통계 정보 (이름, 성별, 생년월일, 주소 등) | 24 |
| `practitioner` | Practitioner | 의료진 정보 (이름, 자격, 성별, 연락처) | 19 |
| `organization` | Organization | 의료기관 정보 (이름, 유형, 주소) | 16 |
| `location` | Location | 의료 서비스 제공 장소 (주소, GPS 좌표) | 21 |
| `practitioner_role` | PractitionerRole | 의료진의 역할 및 전문과목 | 16 |

### 🩺 Clinical (임상) — 14개
| 테이블명 | FHIR 리소스 | 설명 | 컬럼 수 |
|---|---|---|---|
| `encounter` | Encounter | 진료 이력 (방문 유형, 기간, 사유) | 21 |
| `condition` | Condition | 진단 및 건강 상태 (발병일, 임상 상태) | 20 |
| `observation` | Observation | 임상 관찰 (바이탈, 검사 결과, 측정값) | 23 |
| `procedure` | Procedure | 시행된 시술 및 처치 | 16 |
| `allergy_intolerance` | AllergyIntolerance | 알레르기 및 불내증 기록 | 21 |
| `diagnostic_report` | DiagnosticReport | 진단 보고서 (검사 결과, 영상 소견) | 20 |
| `imaging_study` | ImagingStudy | 영상 검사 (X-ray, CT, MRI 등) | 24 |

## Step 10: Test Agent — 환자 검색

In [ ]:
print("Testing: 당뇨 환자 검색")
print("=" * 60)
result = invoke_agent("오늘 외래에 당뇨 진단받은 60대 환자가 있는데, 목록 좀 보여줘.")
print(result)

Testing: 당뇨 환자 검색
총 100명의 당뇨 환자 목록을 정리해 드리겠습니다. 현재 날짜(2025년 기준) 기준으로 **60대(1955~1964년생)** 당뇨 진단 환자 목록입니다.

> ⚠️ 참고: 본 데이터는 Synthea 합성 데이터입니다. "오늘 외래" 필터링은 실시간 방문 데이터와 연계가 필요하며, 아래는 해당 연령대의 당뇨 진단 등록 환자 목록입니다.

---

## 🩺 60대 당뇨 환자 목록 (총 100명 중 주요 환자)

| # | 이름 | 성별 | 생년월일 | 나이(추정) | 연락처 | 거주 도시 |
|---|------|------|----------|-----------|--------|-----------|
| 1 | Noe500 Hermann103 | 남 | 1963-05-22 | 61세 | 555-194-1157 | Waltham |
| 2 | Tinisha932 Effertz744 | 여 | 1958-02-12 | 67세 | 555-621-2896 | Worcester |
| 3 | Daniele610 Schultz619 | 여 | 1961-06-05 | 63세 | 555-441-7074 | Wilbraham |
| 4 | Alphonso102 Gutmann970 | 남 | 1961-02-03 | 64세 | 555-596-5821 | East Longmeadow |
| 5 | Frederic454 Raynor401 | 남 | 1960-08-06 | 64세 | 555-162-8190 | Worcester |
| 6 | Sharon747 Predovic534 | 여 | 1955-05-13 | 69세 | 555-658-8474 | (우편번호 미확인) |
| 7 | Adelle43 Wyman904 | 여 | 1958-03-14 | 67세 | 555-195-3448 | Cohasset |
| 8 | Edwardo860 Kuphal363 | 남 | 1955-01-05 | 70세 | 555-416-3557 | Shrewsbury |
| 9

## Step 11: Test Agent — 인구 건강 분석

In [41]:
print("Testing: 인구 건강 분석")
print("=" * 60)
result = invoke_agent("60-69세 연령대의 당뇨병 환자 인구 건강 지표를 분석해줘")
print(result)

Testing: 인구 건강 분석
조회된 데이터를 바탕으로 상세히 분석해 드리겠습니다.

---

## 📊 60-69세 당뇨병 환자 인구 건강 지표 분석

### 1️⃣ 질환별 환자 현황

| 질환명 | 여성(F) | 남성(M) | 합계 | 비율 |
|---|:---:|:---:|:---:|:---:|
| 당뇨 전단계 (Prediabetes) | 47 | 43 | **90** | 55.9% |
| 제2형 당뇨병 (T2DM) | 12 | 8 | **20** | 12.4% |
| 당뇨병성 신장 질환 | 8 | 10 | **18** | 11.2% |
| 미세알부민뇨 (T2DM) | 8 | 7 | **15** | 9.3% |
| 단백뇨 (T2DM) | 6 | 3 | **9** | 5.6% |
| 비증식성 당뇨병성 망막병증 | 1 | 3 | **4** | 2.5% |
| 당뇨병성 신경병증 | 2 | 2 | **4** | 2.5% |
| **전체 합계** | **84** | **76** | **160** | **100%** |

---

### 2️⃣ 성별 분포

| 구분 | 환자 수 | 비율 |
|---|:---:|:---:|
| 여성 (Female) | 84명 | 52.5% |
| 남성 (Male) | 76명 | 47.5% |

> 여성이 남성보다 약 **5%p 더 많은** 분포를 보입니다.

---

### 3️⃣ 당뇨 합병증 분석

합병증 환자 수 합계 (제2형 당뇨 제외)를 분석하면:

| 합병증 유형 | 환자 수 | 합병증 비율* |
|---|:---:|:---:|
| 🫘 당뇨병성 신장 질환 | 18명 | 56.3% |
| 🔬 미세알부민뇨 | 15명 | 46.9% |
| 🧪 단백뇨 | 9명 | 28.1% |
| 👁️ 당뇨병성 망막병증 | 4명 | 12.5% |
| 🦵 당뇨병성 신경병증 | 4명 | 12.5% |

*\*T2DM 환자 32명 대비 비율 (중복 진단 포함)*

---

### 4️⃣ 주요 인사이트

#### ⚠️ 위험 신호
- **당뇨 전단계 환자가

## Step 12: Test Memory — 세션 내 기억

In [42]:
import uuid
session_id = str(uuid.uuid4()) + "-" + "a" * 10

r1 = invoke_agent("나는 내분비내과 전문의이고, 주로 당뇨병 환자를 담당하고 있어", session_id)
print("Message 1:", r1[:500])

r2 = invoke_agent("내가 담당하는 질환의 환자 통계를 보여줘", session_id)
print("\nMessage 2:", r2[:500])

Message 1: 안녕하세요! 내분비내과 전문의 선생님, 반갑습니다. 😊

저는 FHIR 기반 의료 데이터 레이크를 조회하여 당뇨병 환자 관련 정보를 분석해 드릴 수 있는 AI 에이전트입니다.

선생님께서 주로 필요하실 만한 기능들을 안내해 드리겠습니다:

---

## 🩺 활용 가능한 주요 기능

### 👤 환자 관리
- **특정 환자 조회** — 이름이나 환자 ID로 환자 검색 및 종합 요약
- **당뇨병 환자 목록** — 당뇨 진단 코드로 환자 검색
- **진단/처방 이력** — 특정 환자의 진단 이력 및 현재 복용 약물 확인

### 📊 임상 데이터 분석
- **혈당/HbA1c 추이** — 검사 결과 및 바이탈 조회
- **케어 갭 탐지** — 누락된 검사, 미완료 케어플랜 확인
- **진료 이력** — 외래/입원 기록 조회

### 📈 인구 건강 분석
- **당뇨 환자 통계** — 연령대, 성별별 당뇨 환자 현황
- **맞춤형 쿼리** — 복잡한 분석을 위한 SQL 직접 실행

---

##

Message 2: 어떤 질환을 담당하고 계신지 알려주시면 해당 질환의 환자 통계를 조회해 드리겠습니다! 😊

예를 들어 아래와 같이 말씀해 주세요:

- **"당뇨병 환자 통계 보여줘"**
- **"고혈압 환자 통계 보여줘"**
- **"심부전 환자 통계 보여줘"**

또는 특정 **SNOMED/ICD 코드**가 있으시면 함께 알려주세요.

담당 질환명을 입력해 주시면 바로 조회해 드리겠습니다! 🏥


## Step 13: Observability Dashboard

In [43]:
print(f"📊 GenAI Observability Dashboard:")
print(f"   https://console.aws.amazon.com/cloudwatch/home?region={REGION}#gen-ai-observability/agent-core")

📊 GenAI Observability Dashboard:
   https://console.aws.amazon.com/cloudwatch/home?region=us-west-2#gen-ai-observability/agent-core


## Step 14: Run Streamlit Frontend

AgentCore Runtime에 배포된 에이전트를 호출하는 Streamlit UI를 실행합니다.

In [44]:
print(f"Agent Runtime ARN: {AGENT_RUNTIME_ARN}")
print()
print("Streamlit을 실행하려면 터미널에서 다음 명령어를 실행하세요:")
print()
print(f'export AGENT_ARN="{AGENT_RUNTIME_ARN}"')
print(f'export AWS_REGION="{REGION}"')
print('cd /workshop/building-medical-agent-from-ai-ready-datalake/agent')
print('pip install -q streamlit boto3')
print('streamlit run app.py --server.port 8501 --server.address 0.0.0.0')
print()
print("⚠️  EC2 보안그룹에서 8501 포트 인바운드를 허용해야 합니다.")

Agent Runtime ARN: arn:aws:bedrock-agentcore:us-west-2:997924005000:runtime/fhir_medical_agent-UWvFdU9j5I

Streamlit을 실행하려면 터미널에서 다음 명령어를 실행하세요:

export AGENT_ARN="arn:aws:bedrock-agentcore:us-west-2:997924005000:runtime/fhir_medical_agent-UWvFdU9j5I"
export AWS_REGION="us-west-2"
cd /workshop/building-medical-agent-from-ai-ready-datalake/agent
pip install -q streamlit boto3
streamlit run app.py --server.port 8501 --server.address 0.0.0.0

⚠️  EC2 보안그룹에서 8501 포트 인바운드를 허용해야 합니다.


In [48]:
# 보안그룹에 8501 포트 추가 (접속자 IP만 허용)
ec2 = boto3.client('ec2', region_name=REGION)

my_ip = input('브라우저에서 https://checkip.amazonaws.com 접속 후 표시된 IP를 입력하세요: ').strip()

instances = ec2.describe_instances(
    Filters=[{'Name': 'tag:Name', 'Values': ['*CodeEditor*']}, {'Name': 'instance-state-name', 'Values': ['running']}]
)['Reservations']

if instances:
    sg_id = instances[0]['Instances'][0]['SecurityGroups'][0]['GroupId']
    public_ip = instances[0]['Instances'][0].get('PublicIpAddress', 'N/A')
    try:
        ec2.authorize_security_group_ingress(
            GroupId=sg_id,
            IpPermissions=[{'IpProtocol': 'tcp', 'FromPort': 8501, 'ToPort': 8501, 'IpRanges': [{'CidrIp': f'{my_ip}/32', 'Description': 'Streamlit UI'}]}]
        )
        print(f'✅ 보안그룹 {sg_id}에 8501 포트 추가 완료 ({my_ip}/32)')
    except Exception as e:
        if 'Duplicate' in str(e):
            print(f'✅ 8501 포트 이미 허용됨')
        else:
            print(f'⚠️ {e}')
    print(f'\n🌐 Streamlit URL: http://{public_ip}:8501')
else:
    print('CodeEditor 인스턴스를 찾을 수 없습니다.')

✅ 보안그룹 sg-00450fa5233ebacae에 8501 포트 추가 완료 (15.248.4.242/32)

🌐 Streamlit URL: http://54.218.85.210:8501


## Streamlit 구동 스크립트 실행 
VS Code 터미널에서 /workshop/building-medical-agent-from-ai-ready-datalake/run_app.sh 스크립트를 실행합니다.

## Step 15: Save Agent Info

In [46]:
agent_info = {
    "agent_runtime_arn": AGENT_RUNTIME_ARN,
    "agent_runtime_id": AGENT_RUNTIME_ID,
    "memory_id": MEMORY_ID,
    "gateway_mcp_url": GATEWAY_MCP_URL,
    "region": REGION,
}
with open("agent_info.json", "w") as f:
    json.dump(agent_info, f, indent=2)
print("✅ Saved to agent_info.json")
print(json.dumps(agent_info, indent=2))

✅ Saved to agent_info.json
{
  "agent_runtime_arn": "arn:aws:bedrock-agentcore:us-west-2:997924005000:runtime/fhir_medical_agent-UWvFdU9j5I",
  "agent_runtime_id": "fhir_medical_agent-UWvFdU9j5I",
  "memory_id": "fhir_agent_mem_047e137d-3CWGJO9MS3",
  "gateway_mcp_url": "https://fhir-medical-gateway-zrar3kofie.gateway.bedrock-agentcore.us-west-2.amazonaws.com/mcp",
  "region": "us-west-2"
}


## Cleanup (Optional)

In [ ]:
# ⚠️ Uncomment to delete resources
# agentcore.delete_agent_runtime_endpoint(agentRuntimeId=AGENT_RUNTIME_ID, endpointName="DEFAULT")
# agentcore.delete_agent_runtime(agentRuntimeId=AGENT_RUNTIME_ID)
# agentcore.delete_memory(memoryId=MEMORY_ID)
# iam_client.delete_role_policy(RoleName=RUNTIME_ROLE_NAME, PolicyName="AgentCoreRuntimePolicy")
# iam_client.delete_role(RoleName=RUNTIME_ROLE_NAME)
# print("All agent resources deleted")